In [5]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords , shakespeare

In [6]:

nltk.download('shakespeare')
nltk.download('stopwords')

[nltk_data] Downloading package shakespeare to
[nltk_data]     C:\Users\Yatharth_Shivam\AppData\Roaming\nltk_data...
[nltk_data]   Package shakespeare is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Yatharth_Shivam\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [7]:
# TILL NOW I HAVE CREATED A END TO END TFIDF SEARCH ENGINE BUT IT IS INEFFICIENT 
## RN EVERY VECTOR IS SPARSE IN NATURE TRULY AS EACH WORD IS STORED AS DIMS BUT IN THIS OPTIMIZED VERSION WE ONLY STORE NON ZERO ENTRIES

#   1. THE DOCS VECTORS ARE SPARSE IN NATURE 
#   2. THE QUERY VECTOR IS ALSO SPARSE IN NATURE
#   3. THE SCORES ARE VERY SMALL (EVEN IF QUERY MATCHES THE ACTUAL WORDS IN DOCS ) REASON IS COSINE DEPENDS ON THE DIMS OF THE VECTORS
#       AND SPARSITY SO SPARSE VECTORS HAVE 0 WHICH AFFECTS THE COSINE SIMILARITY

In [8]:
# TO OPTIMIZE THE BUILDING OF THE INV_IDX WE CREATE A NEW IDX THAT IS TRANPOSE OF THE INV_IDX
# INV_IDX : {WORD : {DOC_ID: FREQ}}  
# FOR_IDX : {DOC_ID : {WORD : FREQ}}
# THIS IS DONE BECAUSE IN SEARCHING TIME WE NEED BOTH INV_IDX AND FORWARD_IDX AS THE VECTORS ARE CREATED AND ASSIGNED TO A DOC_ID NOT A WORD
#  WE INIT FORWARD_IDX OUTSIDE THE LOOP 
#  WE INIT THE FORWARD_IDX FOR A DOC_ID INSIDE FIRST LOOP OUTSIDE SECOND LOOP AS THE SECOND LOOP , ITERATES OVER WORDS IN THAT DOC_ID TILL WORDS ARE
#  SINCE FORWARD_IDX KEY IS DOC_ID SO THE NUMS OF KEYS SHOULD BE SAME AS THAT OF DOCS IN THE CORPUS WHICH THE FIRST LOOP GUARENTEES 

In [10]:
def term_freq_unnorm_idx(corpus_name,lang):
    inverted_index = {}
    forward_index = {}
    file_lst = corpus_name.fileids()
    stopword = set(stopwords.words(lang))
    for id , doc in enumerate(file_lst):
        raw_words = corpus_name.words(fileid = doc)
        pro_words = [word.lower() for word in raw_words if word.lower() not in stopword]
        forward_index[id] = {}
        for word in pro_words:
            if word not in inverted_index : inverted_index[word] = {}
            inverted_index[word][id] = inverted_index[word].get(id,0) + 1
            
            forward_index[id][word] = forward_index[id].get(id,0) + 1
        
            
    return inverted_index ,forward_index, file_lst
        
inv_idx , for_idx , corpus = term_freq_unnorm_idx(shakespeare,'english')       


In [ ]:
inv_idx

In [ ]:
for_idx

In [13]:
import math
def tf_(raw_counts):
    if raw_counts == 0 : return 0 
    return 1 + math.log10(raw_counts)

def idf_(word, inv_idx, file_lst):
    N = len(file_lst)
    df = len(inv_idx[word])
    return math.log10(N / df)


In [ ]:

# Creating a new tf_idf_ func that uses forward index also
# using dict.get(word,0) to add a safety layer if word freq is 0 returns 0

In [52]:
def tf_idf_(inv_idx,for_idx ,word,doc_id,corpus_lst):
    if word not in inv_idx : return 0
    if doc_id not in for_idx : return 0
    if doc_id not in inv_idx[word] : return 0
    raw_counts = for_idx[doc_id].get(word,0)
    if raw_counts == 0 : return 0
    tf = tf_(raw_counts)
    idf = idf_(word,inv_idx,corpus_lst)
    return tf*idf
    

In [ ]:
# The real use of forward index now comes in vectorizing :
# 1. Before we were using inv_idx and iterating through all vocab words and checking if this word appears in a specific doc_id and how many times
# Now we have for_idx so we only iterate through words in that specifi doc_id not all the vocab words

In [51]:
def vectorize_one_doc(doc_ids,inv_idx,for_idx,corpus):
    one_doc_vec = {}
    if doc_ids not in for_idx : return {}
    for word in for_idx[doc_ids]:
        one_doc_vec[word] = tf_idf_(inv_idx,for_idx,word,doc_ids,corpus)
        
    return one_doc_vec
        
    

In [53]:
def vectorize_all_docs(inv_idx,for_idx,corpus):
    all_doc_vec = {}
    for doc_id in range(len(corpus)):
        all_doc_vec[doc_id] = vectorize_one_doc(doc_id , inv_idx,for_idx,corpus)
    return all_doc_vec
        

In [54]:
def vectorize_query(query,inv_idx,corpus):
    pro_query = query.lower().split()
    query_vec = {}
    
    for word in set(pro_query) :
        if word not in inv_idx : continue
        raw_counts = pro_query.count(word)
        tf = tf_(raw_counts)
        idf = idf_(word,inv_idx,corpus)
        query_vec[word] = tf*idf
        
    return query_vec
    

In [55]:
# One more optimization is add a get_candidate func that narrows the cosine similarity processing 
# Simply search words that are in the inv_idx that are present in the query vector
# query vect : { word : tfidf }
# search this word in inv_idx and add the id in the set 


In [56]:
def get_candidate(query_vec,inv_idx):
    cand = set()
    for word in query_vec:
        cand |= set(inv_idx[word].keys())
    return cand

In [57]:
# We can also create a func that computes the docs magnitude once and cache it for future uses

In [58]:
def precompute_doc_magnitudes(all_docs_vec):
    return {
        doc_id: math.sqrt(sum(v ** 2 for v in vec.values()))
        for doc_id, vec in all_docs_vec.items()
    }


In [59]:

def search_vector(query_vec,doc_mags, all_docs_vec,inv_idx):
    cands = get_candidate(query_vec,inv_idx)
    query_mag = math.sqrt(sum(val ** 2 for val in query_vec.values()))
    scores = {}
    for doc_id in cands:
        doc_vec = all_docs_vec[doc_id]
        dot = sum(query_vec[w]*doc_vec[w] for w in query_vec if w in doc_vec)
        doc_mag = doc_mags[doc_id]
        scores[doc_id] = dot / (doc_mag*query_mag) if doc_mag and query_mag else 0
    return scores

In [60]:

def rank_results(scores, corpus, k=10):
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return {corpus[doc_id]: score for doc_id, score in ranked[:k]}


all_docs_vec = vectorize_all_docs(inv_idx, for_idx,corpus)
doc_magnitudes = precompute_doc_magnitudes(all_docs_vec)


In [62]:
print(all_docs_vec[1])
print(vectorize_query('macbeth and caesar',inv_idx,corpus))
print(for_idx[0])
print(vectorize_one_doc(0,inv_idx,for_idx,corpus))

{'midsummer': 0.9030899869919435, 'night': 0.0, "'": 0.0, 'dream': 0.05799194697768673, 'dramatis': 0.0, 'personae': 0.0, 'theseus': 0.9030899869919435, ',': 0.0, 'duke': 0.3010299956639812, 'athens': 0.6020599913279624, '.': 0.0, 'egeus': 0.9030899869919435, 'father': 0.0, 'hermia': 0.9030899869919435, 'lysander': 0.9030899869919435, 'demetrius': 0.6020599913279624, 'love': 0.0, 'philostrate': 0.9030899869919435, 'master': 0.0, 'revels': 0.12493873660829993, 'quince': 0.9030899869919435, 'carpenter': 0.4259687322722811, 'snug': 0.9030899869919435, 'joiner': 0.6020599913279624, 'bottom': 0.3010299956639812, 'weaver': 0.9030899869919435, 'flute': 0.9030899869919435, 'bellows': 0.6020599913279624, '-': 0.0, 'mender': 0.6020599913279624, 'snout': 0.9030899869919435, 'tinker': 0.9030899869919435, 'starveling': 0.9030899869919435, 'tailor': 0.2041199826559248, 'hippolyta': 0.9030899869919435, 'queen': 0.12493873660829993, 'amazons': 0.9030899869919435, 'betrothed': 0.9030899869919435, 'daug

In [63]:
def search(query, inv_idx, all_docs_vec, doc_magnitudes, corpus, k=10):
    query_vec = vectorize_query(query, inv_idx, corpus)
    scores = search_vector(query_vec, doc_magnitudes,all_docs_vec, inv_idx)
    return rank_results(scores, corpus, k)


# usage:
print(search('macbeth and caesar', inv_idx, all_docs_vec, doc_magnitudes, corpus))

{'macbeth.xml': 0.0329760224532347, 'j_caesar.xml': 0.001853420527685849, 'othello.xml': 0.0014823607224180829, 'a_and_c.xml': 0.0013967172350598956, 'hamlet.xml': 0.0012019802941113573}
